# Pipeline completo: Preprocesamiento + Filtrado + Modelo + Optimización + Calibración + Persistencia + Inferencia

Este notebook implementa el flujo **end-to-end** de un modelo de scoring de crédito sobre el dataset de Lending Club. La idea es **encadenar todas las piezas** que se han ido construyendo a lo largo del curso en un único pipeline reproducible:

1. **Preprocesamiento** (`BasePreprocess`): nulls, fechas, OHE de categóricas, *embeddings* de texto, transformación cuantil de numéricas y *features* cruzadas (polinomio de grado 2 con interacciones).
2. **Filtrado de features** (`BaseFiltering`): elimina constantes, correlacionadas y peores que ruido aleatorio.
3. **Modelo base**: `CatBoostClassifier` con `auto_class_weights='Balanced'` (clases muy desbalanceadas).
4. **Optimización de hiperparámetros**: Optuna con sampler **TPE** y `MedianPruner`, optimizando **Brier score** sobre el set de validación.
5. **Diagnóstico de calibración**: test de **Spiegelhalter** sobre las probabilidades crudas.
6. **Calibración post-hoc**: `VennAbersCalibrator` (cross-Venn-Abers con 3 splits), que además devuelve un **intervalo `[p_low, p_high]`** que cuantifica la incertidumbre epistémica de la probabilidad.
7. **Persistencia** de los tres artefactos en `pkl` (preprocesador, filtro y calibrador). Estos tres objetos son **todo lo que necesita producción** para puntuar un crédito nuevo.
8. **Inferencia sobre test**: cargamos los `pkl` y reproducimos el flujo `transform → transform → predict_proba` sobre `df_test_small.csv` para validar que el pipeline serializado funciona como esperamos.

> **Idea clave**: el preprocesador y el filtro hacen `fit` **solo con train**. En test (y en producción) se aplica únicamente `transform`, que reutiliza los parámetros aprendidos. Así evitamos *data leakage* y garantizamos que la distribución de features que entra al modelo en inferencia es la misma que vio durante el entrenamiento.

## PASO 1 — Preprocesamiento de datos

`BasePreprocess` encapsula toda la lógica que convierte el CSV crudo en una matriz numérica que un modelo puede consumir. Internamente:

- **Selecciona variables candidatas** a partir del Excel `variables_withoutExperts.xlsx` (columna `posible_predictora == 'si'`).
- **Trata nulos**: descarta variables con > 98 % de nulos, imputa con mediana/moda las que tienen < 10 % y rellena con `-1` / `"DESCONOCIDO"` las que están entre 10 % y 98 %.
- **Variables temporales**: extrae año y mes de `earliest_cr_line`.
- **Categóricas de baja cardinalidad** (≤ 50): One-Hot Encoding con `handle_unknown="ignore"` (categorías nuevas en test → todo cero).
- **Texto** (`emp_title`, `desc`): embeddings con `intfloat/e5-small-v2` reducidos a 20 componentes.
- **Numéricas**: `QuantileTransformer` a una distribución normal (robusto a outliers).
- **Features cruzadas**: `PolynomialFeatures(degree=2, interaction_only=True)` sobre las numéricas → mucha más capacidad expresiva sin disparar los OHE.

El patrón es **`fit` solo con train + `transform` para train/test**, igual que cualquier transformador de scikit-learn.

In [61]:
from src.preprocessing.base_preprocessing import BasePreprocess

# Instanciamos la clase de preprocesamiento.
# El fichero Excel contiene la lista de variables candidatas a ser predictoras.
base_pre = BasePreprocess("data/variables_withoutExperts.xlsx", "loan_status")

In [62]:
import pickle
with open('data/filtered/y_train_filtered.pkl', 'rb') as f:
    y_train = pickle.load(f)

In [63]:
# fit(): aprende los parametros del preprocesamiento SOLO con datos de entrenamiento.
# Esto incluye: categorias del OHE, medianas para imputacion, parametros del QuantileTransformer, etc.
base_pre.fit("data/df_train_small.csv")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  self.train_X_data['earliest_cr_line'] = pd.to_datetime(self.train_X_data['earliest_cr_line'])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [64]:
# transform(): aplica las transformaciones aprendidas en fit().
# Devuelve X_train (features) e y_train (target: True=default, False=fully paid).
X_train, y_train = base_pre.transform("data/df_train_small.csv")
print(f"Dimensiones tras preprocesamiento: {X_train.shape[0]} filas x {X_train.shape[1]} columnas")

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:135: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr_line'] = pd.to_datetime(X_data['earliest_cr_line'])


Dimensiones tras preprocesamiento: 80000 filas x 2614 columnas


## PASO 2 — Filtrado de features

Tras el preprocesamiento tenemos ~2.600 features (sobre todo por el polinomio cruzado). Muchas son redundantes o inútiles. `BaseFiltering` aplica **3 filtros secuenciales** que se ejecutan en orden:

1. **`DropConstantFeatures(tol=0.9)`** — elimina features donde el 90 % o más de los valores son iguales. No aportan señal y solo añaden ruido al modelo.
2. **`DropCorrelatedFeatures(threshold=0.8, method='pearson')`** — para cada par de variables con correlación > 0.8, mantiene una y descarta la otra. Reduce la multicolinealidad y acelera el entrenamiento.
3. **`ProbeFeatureSelection(n_probes=10)`** — añade 10 variables aleatorias (*probes* gaussianas) al dataset, entrena un `RandomForest` y descarta toda feature cuya importancia sea menor que la del *probe* más importante. Es un filtro muy potente: si una variable real no le gana a ruido puro, no aporta nada.

> El orden importa: poner `DropCorrelated` después de `DropConstant` evita calcular correlaciones contra columnas constantes (que darían NaN o 0/0). Y poner `ProbeFeatureSelection` al final hace que el RandomForest se entrene sobre un set ya limpio, lo que lo hace mucho más rápido y más fiable.

El `fit` aprende **qué columnas eliminar** mirando solo los datos de train; el `transform` reaplica esa lista a cualquier dataset (train/test/inferencia).

In [65]:
from src.filtering.base_filtering import BaseFiltering

# Instanciamos el filtro con los parametros por defecto.
# Todos los parametros son configurables en el constructor.
base_filter = BaseFiltering(
    constant_tol=0.9,
    correlation_threshold=0.8,
    probe_n_probes=10,
    probe_scoring='roc_auc',
    probe_cv=3,
    probe_n_estimators=50,
    probe_max_depth=10
)

In [66]:
# fit(): aprende que features eliminar usando SOLO datos de train.
# Internamente ejecuta los 3 filtros en secuencia.
base_filter.fit(X_train, y_train)

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs

In [67]:
# Resumen del filtrado: cuantas features se eliminaron en cada paso.
base_filter.print_summary()

RESUMEN DEL PIPELINE DE FILTRADO
  Features iniciales:              2614
  Eliminadas cuasi-constantes:     -133
  Eliminadas por correlacion:       -1837
  Eliminadas por ProbeFeature:      -441
  Features seleccionadas finales:  203


In [68]:
# transform(): aplica los filtros aprendidos en fit() a los datos de train.
X_train_filtered = base_filter.transform(X_train)

print(f"Features seleccionadas ({X_train_filtered.shape[1]}):")
print(X_train_filtered.columns.tolist())

Features seleccionadas (203):
['verification_status_Not Verified', 'verification_status_Verified', 'term_ 36 months', 'emp_title_00', 'emp_title_01', 'emp_title_02', 'emp_title_04', 'emp_title_06', 'emp_title_08', 'emp_title_10', 'emp_title_13', 'emp_title_19', 'loan_amnt', 'annual_inc', 'dti', 'loan_amnt annual_inc', 'loan_amnt dti', 'loan_amnt inq_last_6mths', 'loan_amnt mths_since_last_delinq', 'loan_amnt open_acc', 'loan_amnt revol_bal', 'loan_amnt revol_util', 'loan_amnt all_util', 'loan_amnt acc_open_past_24mths', 'loan_amnt avg_cur_bal', 'loan_amnt bc_open_to_buy', 'loan_amnt bc_util', 'loan_amnt mo_sin_old_il_acct', 'loan_amnt mo_sin_rcnt_rev_tl_op', 'loan_amnt mo_sin_rcnt_tl', 'loan_amnt mort_acc', 'loan_amnt mths_since_recent_bc', 'loan_amnt mths_since_recent_inq', 'loan_amnt num_actv_bc_tl', 'loan_amnt num_actv_rev_tl', 'loan_amnt num_il_tl', 'loan_amnt num_op_rev_tl', 'loan_amnt num_tl_op_past_12m', 'loan_amnt percent_bc_gt_75', 'loan_amnt total_bal_ex_mort', 'loan_amnt tot

### Persistencia parcial: preprocesador y filtro

Guardamos `base_pre` y `base_filter` ya **fiteados** como `pkl`. Estos dos objetos contienen *todo* el estado aprendido del entrenamiento (categorías OHE, medianas, parámetros del `QuantileTransformer`, lista de columnas a eliminar…). Más adelante los recargaremos para hacer inferencia sobre test sin necesidad de volver a ejecutar `fit`.

In [69]:
# save base_pre como un objeto pikle usando la libreia pickle
import pickle
with open("base_pre.pkl", "wb") as f:
    pickle.dump(base_pre, f)

# save base_filter como un objeto pikle usando la libreia pickle
with open("base_filter.pkl", "wb") as f:
    pickle.dump(base_filter, f)

## PASO 3 — Modelo: CatBoost + optimización con Optuna (TPE)

Usamos `CatBoostClassifier` como modelo base. CatBoost funciona muy bien *out-of-the-box* en problemas tabulares con muchas features y clases desbalanceadas, y trae **`BrierScore` como métrica nativa**, lo que nos permite optimizar directamente la métrica que nos importa para calibración.

**Estrategia de splits:**
- Partimos `(X_train_filtered, y_train)` en `train` (60 %), `validation` (20 %) y **`calibration` (20 %)**.
- `train` → ajusta el modelo.
- `validation` → guía a Optuna (early stopping y métrica del trial).
- `calibration` → **nunca ve el modelo durante el tuning**; se reserva en exclusiva para diagnosticar y entrenar el calibrador `Venn-Abers`.

**Optuna:**
- `TPESampler(multivariate=True)` — explora el espacio de hiperparámetros aprendiendo de los trials previos.
- `MedianPruner` — corta los trials que en las primeras 50 iteraciones ya están por debajo de la mediana de los anteriores; ahorra muchísimo tiempo.
- `CatBoostPruningCallback('BrierScore')` — comunica a Optuna la métrica intermedia para que pueda decidir si podar.

In [70]:
# antes vamos a hacer un split  y_train y X_train_filtered en train y calibration
from sklearn.model_selection import train_test_split
y_train_flat = y_train.values.ravel()
X_tr, X_cal, y_tr, y_cal = train_test_split(
    X_train_filtered, y_train_flat,
    test_size=0.2, random_state=42, stratify=y_train_flat
)
# parto train en train y valodacion
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr,
    test_size=0.25, random_state=42, stratify=y_tr
)
 

In [71]:
from catboost import CatBoostClassifier
from optuna.integration import CatBoostPruningCallback
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.integration import LightGBMPruningCallback
from scipy.special import expit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, roc_auc_score, roc_curve, brier_score_loss
)

def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 5e-3, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 100),
        'bootstrap_type': 'Bernoulli',
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'auto_class_weights': 'Balanced',
        'eval_metric': 'BrierScore',       # CatBoost tiene Brier nativo
        'random_seed': 42,
        'early_stopping_rounds': 50,
        'verbose': False,
    }

    model = CatBoostClassifier(**params)
    pruning_cb = CatBoostPruningCallback(trial, 'BrierScore')

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        callbacks=[pruning_cb],
    )
    pruning_cb.check_pruned()

    prob_val = model.predict_proba(X_val)[:, 1]
    return brier_score_loss(y_val, prob_val)

In [72]:
study_catboost = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(n_startup_trials=3, multivariate=True, seed=42),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=50),
    study_name='catboost_tpe_brier',
)

study_catboost.optimize(objective_catboost, n_trials=10, show_progress_bar=True)

print(f"\nMejor Brier en val: {study_catboost.best_value:.4f}")
print("Mejores hiperparametros:")
for k, v in study_catboost.best_params.items():
    print(f"  {k}: {v}")




/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/optuna/_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-04-30 21:07:11,296] A new study created in memory with name: catboost_tpe_brier


  0%|          | 0/10 [00:00<?, ?it/s]

/var/folders/_0/0zwytxv51nb3c6wxtthk3bwh0000gn/T/ipykernel_19677/756382244.py:32: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_cb = CatBoostPruningCallback(trial, 'BrierScore')


[I 2026-04-30 21:07:20,029] Trial 0 finished with value: 0.203043514180067 and parameters: {'learning_rate': 0.023171758042400858, 'depth': 10, 'l2_leaf_reg': 0.847180141881998, 'min_data_in_leaf': 62, 'subsample': 0.5780093202212182, 'colsample_bylevel': 0.5779972601681014, 'random_strength': 0.0017073967431528124}. Best is trial 0 with value: 0.203043514180067.


/var/folders/_0/0zwytxv51nb3c6wxtthk3bwh0000gn/T/ipykernel_19677/756382244.py:32: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_cb = CatBoostPruningCallback(trial, 'BrierScore')


...

[I 2026-04-30 21:08:06,888] Trial 4 finished with value: 0.20398479965276536 and parameters: {'learning_rate': 0.0058364361770731135, 'depth': 10, 'l2_leaf_reg': 1.289460667186976, 'min_data_in_leaf': 83, 'subsample': 0.632852179090312, 'colsample_bylevel': 0.6946610693421076, 'random_strength': 0.008511365559296019}. Best is trial 0 with value: 0.203043514180067.


/var/folders/_0/0zwytxv51nb3c6wxtthk3bwh0000gn/T/ipykernel_19677/756382244.py:32: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_cb = CatBoostPruningCallback(trial, 'BrierScore')


[I 2026-04-30 21:08:18,563] Trial 5 finished with value: 0.20779651908840246 and parameters: {'learning_rate': 0.024220237524100515, 'depth': 8, 'l2_leaf_reg': 0.09269273920198684, 'min_data_in_leaf': 16, 'subsample': 0.5408773954239052, 'colsample_bylevel': 0.7030487845127644, 'random_strength': 0.007571235210485213}. Best is trial 0 with value: 0.203043514180067.


/var/folders/_0/0zwytxv51nb3c6wxtthk3bwh0000gn/T/ipykernel_19677/756382244.py:32: ExperimentalWarning: CatBoostPruningCallback is experimental (supported from v3.0.0). The interface can change in the future.
  pruning_cb = CatBoostPruningCallback(trial, 'BrierScore')


[I 2026-04-30 21:08:19,162] Trial 6 pruned. Trial was pruned at iteration 50.
[I 2026-04-30 21:08:21,907] Trial 7 pruned. Trial was pruned at iteration 50.
[I 2026-04-30 21:08:24,277] Trial 8 pruned. Trial was pruned at iteration 50.
[I 2026-04-30 21:08:27,055] Trial 9 pruned. Trial was pruned at iteration 50.

Mejor Brier en val: 0.2030
Mejores hiperparametros:
  learning_rate: 0.023171758042400858
  depth: 10
  l2_leaf_reg: 0.847180141881998
  min_data_in_leaf: 62
  subsample: 0.5780093202212182
  colsample_bylevel: 0.5779972601681014
  random_strength: 0.0017073967431528124


In [73]:
# CatBoost final con mejores params
best_params_cb = dict(study_catboost.best_params)
best_params_cb.update({
    'iterations': 2000,
    'bootstrap_type': 'Bernoulli',
    'auto_class_weights': 'Balanced',
    'eval_metric': 'BrierScore',
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': False,
})

catboost_best = CatBoostClassifier(**best_params_cb)
catboost_best.fit(X_tr, y_tr, eval_set=(X_val, y_val))

CatBoostClassifier(auto_class_weights='Balanced', bootstrap_type='Bernoulli', colsample_bylevel=0.5779972601681014, depth=10, early_stopping_rounds=50, eval_metric='BrierScore', iterations=2000, l2_leaf_reg=0.847180141881998, learning_rate=0.023171758042400858, min_data_in_leaf=62, random_seed=42, random_strength=0.0017073967431528124, subsample=0.5780093202212182, verbose=False)

In [74]:
# Calculamos probabilidades con nuestro set de calibracion
y_prob = catboost_best.predict_proba(X_cal)[:, 1]

## PASO 4 — Diagnóstico de calibración: test de Spiegelhalter

Antes de calibrar, hay que **medir** si las probabilidades de CatBoost ya están calibradas. Si lo estuvieran, calibrar las dañaría más que ayudaría.

**Test de Spiegelhalter (1986):**
- **H0**: el modelo está calibrado, es decir, $\\hat{p}_i = P(Y_i = 1 \\mid X_i)$.
- **H1**: el modelo no está calibrado.

Bajo H0, el Brier score observado debe coincidir con el Brier esperado $\\mathbb{E}[\\hat{p}(1-\\hat{p})]$. La discrepancia entre ambos, normalizada por su desviación estándar bajo H0, sigue una **N(0,1)** asintóticamente. Si $|Z| > 1.96$ rechazamos calibración al 5 %.

Esto es más informativo que un *reliability diagram* visual: nos da un test estadístico formal con un p-valor.

In [75]:
import numpy as np
def spiegelhalter_z_test(y_true, y_prob):
    """
    Test de Spiegelhalter (1986) para calibracion.

    H0: el modelo esta calibrado (p_i = P(Y=1|X=x_i))
    H1: el modelo NO esta calibrado

    Returns: Z-statistic, Brier observado, Brier esperado bajo H0
    """
    N = len(y_true)

    # Brier score observado
    B_obs = np.mean((y_true - y_prob) ** 2)

    # Brier esperado bajo calibracion perfecta
    B_exp = np.mean(y_prob * (1 - y_prob))

    # Varianza bajo H0
    Var_B = (1 / N**2) * np.sum((1 - 2 * y_prob)**2 * y_prob * (1 - y_prob))

    # Z-estadistico
    Z = (B_obs - B_exp) / np.sqrt(Var_B)

    return Z, B_obs, B_exp


Z_stat, B_obs, B_exp = spiegelhalter_z_test(y_cal, y_prob)

In [76]:
print("=" * 60)
print("TEST DE SPIEGELHALTER")
print("=" * 60)
print(f"\n  Brier observado:    {B_obs:.6f}")
print(f"  Brier esperado H0:  {B_exp:.6f}")
print(f"  Diferencia:         {B_obs - B_exp:.6f}")
print(f"\n  Z-estadistico:      {Z_stat:.4f}")
print(f"  Umbral (alpha=5%):  1.96")
print(f"  |Z| > 1.96?         {'SI' if abs(Z_stat) > 1.96 else 'NO'}")

if abs(Z_stat) > 1.96:
    print(f"\n  >>> RESULTADO: Rechazamos H0. Las probabilidades de CatBoost")
    print(f"      NO estan calibradas (|Z| = {abs(Z_stat):.2f} >> 1.96).")
    print(f"      El modelo NECESITA calibracion post-hoc.")
else:
    print(f"\n  >>> RESULTADO: No rechazamos H0. Las probabilidades de CatBoost")
    print(f"      estan razonablemente calibradas (|Z| = {abs(Z_stat):.2f} < 1.96).")

TEST DE SPIEGELHALTER

  Brier observado:    0.204211
  Brier esperado H0:  0.223221
  Diferencia:         -0.019010

  Z-estadistico:      -17.3182
  Umbral (alpha=5%):  1.96
  |Z| > 1.96?         SI

  >>> RESULTADO: Rechazamos H0. Las probabilidades de CatBoost
      NO estan calibradas (|Z| = 17.32 >> 1.96).
      El modelo NECESITA calibracion post-hoc.


## PASO 5 — Calibración con Venn-Abers

El test rechaza H0 con $|Z| \\gg 1.96$, así que las probabilidades de CatBoost **necesitan calibración**. Aplicamos `VennAbersCalibrator` en su variante **cross-Venn-Abers** (`inductive=False, n_splits=3`):

- Es **distribution-free**: no asume ninguna forma paramétrica para la curva de calibración (a diferencia de Platt).
- Es **isotónico bajo el capó** pero con una garantía teórica: la probabilidad real está dentro del intervalo $[p_{low}, p_{high}]$ con probabilidad 1, **bajo el supuesto de intercambiabilidad** de los datos.
- Devuelve una **probabilidad puntual** (la media de $p_{low}$ y $p_{high}$) **y un intervalo de incertidumbre**, lo que es muy útil para decisiones de negocio (p. ej., enviar a revisión manual los créditos cuyo intervalo cruza el umbral de decisión).

`fit(X_cal, y_cal)` entrena el calibrador en el set de calibración reservado al inicio.

In [77]:
from venn_abers import VennAbersCalibrator
va_cal = VennAbersCalibrator(estimator=catboost_best, inductive=False, n_splits=3)
va_cal.fit(X_cal, y_cal)

,estimator,CatBoostClass...verbose=False)
,inductive,False
,n_splits,3
,cal_size,None
,train_proper_size,None
,random_state,None
,shuffle,True
,stratify,None
,precision,None
,cv_ensemble,True


### Persistencia del calibrador

Guardamos el `VennAbersCalibrator` ya ajustado. Importante: el objeto serializa también una **referencia al `catboost_best`** subyacente, así que con un solo `pkl` (`va_cal.pkl`) tenemos a la vez el modelo y su capa de calibración.

In [78]:
# guardamos el calibrador Venn-Abers como pickle
with open("va_cal.pkl", "wb") as f:
    pickle.dump(va_cal, f)


### Inspección rápida: probabilidades calibradas con intervalo

`predict_proba(..., p0_p1_output=True)` devuelve dos elementos:

- La probabilidad puntual calibrada (lo que normalmente se usa para puntuar).
- El **par `(p0, p1)`** sobre los `n_splits=3` calibradores del cross-Venn-Abers; promediando se obtiene `p_low` y `p_high`, los **límites del intervalo** de incertidumbre.

Lo aplicamos a 2 ejemplos de `X_cal` para ver la salida.

In [79]:
X_cal.shape[1:2]

(203,)

In [80]:
_,p0p1 = va_cal.predict_proba(X_cal[200:202], p0_p1_output=True)
p0p1 = np.asarray(p0p1)
p_low  = p0p1[:, :, 0].mean(axis=0)
p_high = p0p1[:, :, 1].mean(axis=0)
p_low, p_high



/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/venn_abers/venn_abers.py:111: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:


(array([0.2455516, 0.2038835]), array([0.23076923, 0.22306717]))

---

## PASO 6 — Inferencia end-to-end con los artefactos serializados

Hasta aquí hemos *entrenado* el pipeline en memoria. En producción no haríamos `fit` nunca: simplemente cargaríamos los `pkl` y aplicaríamos `transform` / `predict_proba`. Vamos a simular ese escenario para validar que el pipeline funciona sobre datos que **el modelo nunca ha visto** (`df_test_small.csv`).

El flujo de inferencia es exactamente:

```
CSV crudo  ──▶  base_pre.transform  ──▶  base_filter.transform  ──▶  va_cal.predict_proba
   X_test            X_test_pre              X_test_filtered            (prob, [p_low, p_high])
```

Los tres pasos son **stateless en inferencia**: solo leen los parámetros guardados en el `fit` original. Esto es lo que garantiza que la distribución de features que entra al modelo es idéntica a la del entrenamiento.

### 6.1 — Cargar los artefactos serializados

Cargamos los tres `pkl` que guardamos arriba. Tras este bloque tenemos:

- `base_pre_loaded`: `BasePreprocess` con todas las transformaciones aprendidas en train.
- `base_filter_loaded`: `BaseFiltering` con las 196 columnas seleccionadas.
- `va_cal_loaded`: `VennAbersCalibrator` (que internamente lleva el `CatBoostClassifier` óptimo de Optuna).

> Nota: importamos las clases `BasePreprocess` y `BaseFiltering` antes del `pickle.load`. `pickle` necesita poder reconstruir las clases originales para deserializar correctamente.

In [81]:
import pickle

# Importamos las clases para que pickle pueda reconstruir los objetos al cargarlos.
from src.preprocessing.base_preprocessing import BasePreprocess
from src.filtering.base_filtering import BaseFiltering
from venn_abers import VennAbersCalibrator

with open("base_pre.pkl", "rb") as f:
    base_pre_loaded = pickle.load(f)

with open("base_filter.pkl", "rb") as f:
    base_filter_loaded = pickle.load(f)

with open("va_cal.pkl", "rb") as f:
    va_cal_loaded = pickle.load(f)

print("Preprocesador cargado :", type(base_pre_loaded).__name__)
print("Filtro cargado        :", type(base_filter_loaded).__name__,
      "| features seleccionadas:", base_filter_loaded.n_features_final)
print("Calibrador cargado    :", type(va_cal_loaded).__name__,
      "| modelo base:", type(va_cal_loaded.estimator).__name__)

Preprocesador cargado : BasePreprocess
Filtro cargado        : BaseFiltering | features seleccionadas: 203
Calibrador cargado    : VennAbersCalibrator | modelo base: CatBoostClassifier


### 6.2 — Lectura del dataset de test

`data/df_test_small.csv` contiene los créditos del **set de test** (~20.000 filas) en su formato crudo, **idéntico al CSV de entrenamiento**: las mismas columnas, los mismos tipos. No tocamos nada manualmente; toda la lógica de limpieza está dentro del preprocesador.

Echamos un primer vistazo para confirmar que el fichero tiene el formato esperado.

In [82]:
import pandas as pd

test_csv_path = "data/df_test_small.csv"
df_test = pd.read_csv(test_csv_path)

print(f"Filas en test : {df_test.shape[0]}")
print(f"Columnas      : {df_test.shape[1]}")
print("\nDistribucion del target (loan_status):")
print(df_test['loan_status'].value_counts(dropna=False))

Filas en test : 20000
Columnas      : 151

Distribucion del target (loan_status):
loan_status
Fully Paid     16003
Charged Off     3997
Name: count, dtype: int64


### 6.3 — Paso 1 del pipeline: `base_pre.transform`

Aplicamos el preprocesador. Internamente:

- Selecciona las mismas columnas predictoras del Excel.
- Imputa nulls con los valores aprendidos en train (mediana, moda, `-1`, `"DESCONOCIDO"`).
- Aplica el `OneHotEncoder` ya fiteado: cualquier categoría que **no estuviera en train** queda como vector cero gracias a `handle_unknown="ignore"`.
- Aplica los `TextEncoder` y el `QuantileTransformer` aprendidos en train.
- Genera las features cruzadas con el `PolynomialFeatures` ya fiteado.

El resultado debe tener **exactamente las mismas 2.614 columnas** que produjimos sobre train.

In [83]:
# Importante: SOLO transform, no fit. El preprocesador ya esta entrenado.
X_test_pre, y_test = base_pre_loaded.transform(test_csv_path)

print(f"Shape tras preprocesamiento: {X_test_pre.shape}")
print(f"Tasa de default en test    : {y_test.values.ravel().mean():.4f}")
X_test_pre.head(3)

/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/src/preprocessing/base_preprocessing.py:135: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  X_data['earliest_cr_line'] = pd.to_datetime(X_data['earliest_cr_line'])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape tras preprocesamiento: (20000, 2614)
Tasa de default en test    : 0.1998


,addr_state_AK,addr_state_AL,addr_state_AR,addr_state_AZ,addr_state_CA,addr_state_CO,addr_state_CT,addr_state_DC,addr_state_DE,addr_state_FL,...,tot_hi_cred_lim total_bal_ex_mort,tot_hi_cred_lim total_bc_limit,tot_hi_cred_lim total_il_high_credit_limit,tot_hi_cred_lim earliest_cr_line_year,total_bal_ex_mort total_bc_limit,total_bal_ex_mort total_il_high_credit_limit,total_bal_ex_mort earliest_cr_line_year,total_bc_limit total_il_high_credit_limit,total_bc_limit earliest_cr_line_year,total_il_high_credit_limit earliest_cr_line_year
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.813073,1.203573,3.233929,-0.396462,1.652737,4.440806,-0.544418,1.899999,-0.232929,-0.625867
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.002570,0.042682,-0.036132,-0.053517,-0.015963,0.013513,0.020015,-0.224453,-0.332450,0.281428
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.679205,0.415887,-3.753571,0.017210,-0.541980,4.891618,-0.022428,-2.995208,0.013733,-0.123947


### 6.4 — Paso 2 del pipeline: `base_filter.transform`

`BaseFiltering.transform` no es más que aplicar las **listas de columnas a eliminar** que aprendió en `fit`:

1. `drop_constant.transform` → quita las cuasi-constantes detectadas en train.
2. `drop_correlated.transform` → quita las correlacionadas detectadas en train.
3. `probe_selection.transform` → se queda con las features que ganaron al ruido en train.

El número de columnas debe coincidir con `n_features_final = 196`. Si no coincidiera, sería señal de que hay un *mismatch* entre el preprocesador serializado y el filtro serializado.

In [84]:
X_test_filtered = base_filter_loaded.transform(X_test_pre)

print(f"Shape tras filtrado: {X_test_filtered.shape}")
print(f"Coincide con n_features_final del filtro? "
      f"{X_test_filtered.shape[1] == base_filter_loaded.n_features_final}")
X_test_filtered.head(3)

Shape tras filtrado: (20000, 203)
Coincide con n_features_final del filtro? True


,verification_status_Not Verified,verification_status_Verified,term_ 36 months,emp_title_00,emp_title_01,emp_title_02,emp_title_04,emp_title_06,emp_title_08,emp_title_10,...,mort_acc mths_since_recent_revol_delinq,mort_acc num_actv_bc_tl,mort_acc num_actv_rev_tl,mort_acc tot_hi_cred_lim,mort_acc total_bc_limit,num_accts_ever_120_pd total_bc_limit,num_actv_rev_tl total_bc_limit,num_bc_sats tot_hi_cred_lim,num_il_tl total_bal_ex_mort,num_tl_op_past_12m total_bc_limit
0,0.0,1.0,1.0,-0.017448,0.155599,0.285106,0.558374,0.180468,0.140769,-0.221423,...,0.038642,-0.054127,-0.073112,-0.046695,-0.027434,-4.372156,1.884485,2.015108,4.290700,1.234333
1,0.0,0.0,0.0,-0.088001,-0.094741,0.387118,0.110625,0.036960,-0.277845,0.028161,...,-0.350754,-0.538669,-0.640977,0.061728,0.383461,-2.677260,-0.443207,-0.042682,0.010298,-2.677260
2,0.0,0.0,1.0,-0.085707,-0.138750,0.631284,-0.431350,-0.010348,0.432002,0.136998,...,-5.708066,0.388073,0.337160,0.792570,0.632441,-2.995208,0.176919,0.650761,-0.046051,-2.995208


### 6.5 — Paso 3 del pipeline: `va_cal.predict_proba`

El último eslabón es el calibrador. `VennAbersCalibrator.predict_proba` internamente:

1. Llama a `catboost_best.predict_proba(X)` para obtener las probabilidades crudas.
2. Las pasa por las funciones isotónicas aprendidas durante el `fit` para devolver probabilidades **calibradas**.
3. Si `p0_p1_output=True`, además devuelve `(p0, p1)` para construir el intervalo de incertidumbre.

Pedimos las dos salidas: las probabilidades calibradas puntuales **y** los límites del intervalo.

In [85]:
import numpy as np

# Probabilidades calibradas puntuales (matriz [P(no-default), P(default)]).
proba_test = va_cal_loaded.predict_proba(X_test_filtered)
p_default = proba_test[:, 1]

# Probabilidades + intervalo p0_p1 (incertidumbre del calibrador).
_, p0p1 = va_cal_loaded.predict_proba(X_test_filtered, p0_p1_output=True)
p0p1 = np.asarray(p0p1)
p_low  = p0p1[:, :, 0].mean(axis=0)
p_high = p0p1[:, :, 1].mean(axis=0)

print(f"Shape de p_default : {p_default.shape}")
print(f"Shape de p_low     : {p_low.shape}")
print(f"Shape de p_high    : {p_high.shape}")
print(f"\nP(default) media en test: {p_default.mean():.4f}")
print(f"Anchura media del intervalo [p_low, p_high]: {(p_high - p_low).mean():.4f}")

Shape de p_default : (20000,)
Shape de p_low     : (20000,)
Shape de p_high    : (20000,)

P(default) media en test: 0.2058
Anchura media del intervalo [p_low, p_high]: -0.0009


/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/venn_abers/venn_abers.py:111: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:
/Users/mmartin/workspaces/education/cunef_ml_2026/modelizacion_datos_2026/.venv/lib/python3.11/site-packages/venn_abers/venn_abers.py:111: RuntimeWarning: All-NaN slice encountered
  if np.sum(np.isnan(np.nanmin(grads))) == 0:


### 6.6 — Resultado: tabla de scoring + métricas en test

Construimos un `DataFrame` con la salida típica que devolvería un servicio de scoring real:

- `p_default` — probabilidad puntual calibrada (lo que se compara contra el umbral de decisión).
- `p_low`, `p_high` — intervalo de Venn-Abers; cuanto más ancho, más insegura es la predicción.
- `width` — anchura del intervalo, útil para enrutar a revisión manual los casos borderline.
- `y_true` — etiqueta real (`True` = default).

Cerramos calculando dos métricas globales en test (**ROC-AUC** y **Brier**) para confirmar que el pipeline cargado desde `pkl` da resultados consistentes con lo que medimos durante el entrenamiento.

In [86]:
from sklearn.metrics import roc_auc_score, brier_score_loss

y_test_flat = y_test.values.ravel().astype(int)

scoring = pd.DataFrame({
    "p_default": p_default,
    "p_low":     p_low,
    "p_high":    p_high,
    "width":     p_high - p_low,
    "y_true":    y_test_flat,
})

print("Primeras 10 predicciones:")
print(scoring.head(10).to_string(index=False))

print("\nMetricas globales en test:")
print(f"  ROC-AUC : {roc_auc_score(y_test_flat, p_default):.4f}")
print(f"  Brier   : {brier_score_loss(y_test_flat, p_default):.4f}")

Primeras 10 predicciones:
 p_default    p_low   p_high     width  y_true
  0.083633 0.080626 0.082278  0.001653       1
  0.351151 0.369458 0.299320 -0.070138       0
  0.115508 0.109422 0.125604  0.016181       1
  0.428136 0.443038 0.409962 -0.033076       0
  0.493745 0.443038 0.538462  0.095424       1
  0.187468 0.200000 0.176991 -0.023009       0
  0.291986 0.292857 0.295699  0.002842       0
  0.260541 0.245552 0.280000  0.034448       0
  0.279359 0.276119 0.280000  0.003881       0
  0.305404 0.308244 0.295699 -0.012545       0

Metricas globales en test:
  ROC-AUC : 0.6809
  Brier   : 0.1495


### Conclusión

Lo que acabamos de demostrar es que con **tres ficheros `.pkl`** (`base_pre.pkl`, `base_filter.pkl`, `va_cal.pkl`) y una sola línea de código por etapa, podemos:

- Recibir un CSV crudo con los créditos a puntuar.
- Aplicar exactamente el mismo preprocesamiento + filtrado que en train (sin riesgo de *data leakage*).
- Devolver una probabilidad **calibrada** y un **intervalo de incertidumbre** por crédito.

Este es el contrato mínimo que necesita producción: el resto del notebook (Optuna, Spiegelhalter, splits) es maquinaria de **entrenamiento** que solo se ejecuta cuando reentrenamos el modelo. Para servir el modelo basta con cargar los tres `pkl` en memoria al arrancar el servicio.